# Priors Diagnostics — Clean, Fast, and Accurate
**Last generated:** 2025-11-09 07:22 UTC  

This notebook provides a complete and lean pipeline for diagnostics of priors for variant forecasting:
- Accurate predictive intervals for counts and allele frequency from a Zero-Inflated Beta-Binomial (ZIBB)
- Randomized PIT with uniformity tests (KS, Chi-square, Cramer-von Mises)
- Coverage metrics across levels and by-decile calibration
- Fit diagnostics (residuals, z-scores, variance ratios, average log-pmf)
- Detection curves (ROC/PR) and LOD (limit of detection) summaries
- Global plots and per-pair PNGs that match typical output structure

Everything is vectorized and chunked for speed on large datasets. All helpers are defined here.

## 0) Configuration
Adjust anything below as needed. Leave `BASE_DIR=None` to auto-detect the repo root (looks for `results/priors/priors_full_detail.csv` up the tree).

In [ ]:

from pathlib import Path
import os, re, json, math, time, warnings
warnings.filterwarnings("ignore", category=UserWarning)

# ========= USER CONFIG (edit as needed) =========
BASE_DIR = None   # e.g., r"C:\Users\<you>\...\oxbio-variant-forecasting"
FIG_DIR_OVERRIDE = None  # e.g., r"C:\...\results\priors\figures"
METRIC_DIR_OVERRIDE = None  # e.g., r"C:\...\results\priors\metric"

# Predictive interval quantiles (allele frequency bands are derived from count bands)
PRED_Q_LO = 0.10
PRED_Q_HI = 0.90

# Plot / output
SAVE_PNG = True
DPI = 140
PAGESIZE = (12, 8)
PLOT_EVERY_NTH = 1  # >1 to subsample points (speed)

# Limits for speed (set None for 'all')
MAX_SITES = None     # e.g., 50
MAX_MUTS  = None     # e.g., 50
MAX_PAIRS = None     # e.g., 100
ROW_LIMIT_GLOBAL = None  # e.g., 400_000

# Sampling caps (speed)
ROW_SAMPLE_PIT       = 120_000
ROW_SAMPLE_COVERAGE  = 120_000
ROW_SAMPLE_KL        = 200_000

# QA thresholds
THRESH = {
    "coverage_tol": 0.06,        # |empirical - nominal| <= 0.06
    "pit_ks_min_p": 0.05,        # KS p >= 0.05
    "vr_min": 0.7, "vr_max": 1.4,
    "mu_drift": 3e-3,            # |slope/day| for mu
    "kappa_drift": 1e-1,         # |slope/day| for kappa
    "kappa_min": 2.0, "kappa_max": 5e3,
}

# Fast/accurate knobs
SEED = 42
CHUNK = 250_000                     # chunk size for heavy ops
HYBRID_INTERVAL = True              # use normal approx for "easy" rows to speed up
EASY_MIN_NK = 40.0                  # use normal approx where n*kappa >= this
EASY_CENTRAL_Q = (0.02, 0.98)       # only approximate for central quantiles
RAND_PIT = True                     # randomized PIT (recommended)
FAST_FLOAT32_AF = True              # store AF bands as float32 to reduce memory
USE_APPROX_THRESHOLD_N = None       # if set (int), use normal approx ppf when n>=N

# Detection & LOD
C_MIN = 3
POWER = 0.95
LOD_USE_MEDIAN_COVERAGE = True

# ================================================

print("[config] Ready.")

## 1) Imports & Environment checks

In [ ]:

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams.update({
    "savefig.bbox": "tight",
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# SciPy is required for accurate ZIBB quantiles / cdf
try:
    from scipy.stats import betabinom, kstest, chisquare
    try:
        from scipy.stats import cramervonmises
    except Exception:
        cramervonmises = None
    from scipy.stats import pearsonr, spearmanr, kendalltau, skew, kurtosis, normaltest
    from scipy.special import erfinv as _erfinv
    HAVE_SCIPY = True
    HAVE_ERFINV = True
except Exception as e:
    HAVE_SCIPY = False
    HAVE_ERFINV = False
    raise SystemExit("SciPy >= 1.7 is required. Please `pip install scipy`.") from e

rng = np.random.default_rng(SEED)
np.seterr(all="ignore")
EPS = 1e-12
print(f"[env] SciPy available: {HAVE_SCIPY}")

## 2) Helpers (paths, filenames, CSV reader)

In [ ]:

_INVALID = re.compile(r'[<>:"/\\|?*\x00-\x1F]')

def safe_fname(x: str, maxlen: int = 200) -> str:
    s = str(x)
    s = _INVALID.sub("_", s).replace(" ", "_").rstrip(". ").strip()
    return s[:maxlen] if maxlen else s

def _read_csv_opt(path: Path, parse_dates=None):
    return pd.read_csv(path, parse_dates=parse_dates) if path.exists() else None

def find_repo_root(start=None) -> Path:
    p = Path(start or os.getcwd()).resolve()
    root_guess = None
    while True:
        if (p / "results" / "priors" / "priors_full_detail.csv").exists():
            return p
        if (p / ".git").exists() or (p / "configs").exists():
            root_guess = p
        if p.parent == p:
            break
        p = p.parent
    return Path(start or os.getcwd()).resolve()

REPO_ROOT = Path(BASE_DIR).resolve() if BASE_DIR else find_repo_root()
PRIORS_DIR = REPO_ROOT / "results" / "priors"
FIG_DIR = Path(FIG_DIR_OVERRIDE) if FIG_DIR_OVERRIDE else (PRIORS_DIR / "figures")
METRIC_DIR = Path(METRIC_DIR_OVERRIDE) if METRIC_DIR_OVERRIDE else (PRIORS_DIR / "metric")
FIG_DIR.mkdir(parents=True, exist_ok=True)
(METRIC_DIR).mkdir(parents=True, exist_ok=True)

CSV_PRIORS = PRIORS_DIR / "priors_full_detail.csv"
assert CSV_PRIORS.exists(), f"Required file not found: {CSV_PRIORS}"

print(f"[paths] REPO_ROOT={REPO_ROOT}")
print(f"[paths] PRIORS={PRIORS_DIR}")
print(f"[paths] FIGURES -> {FIG_DIR}")
print(f"[paths] METRIC  -> {METRIC_DIR}")

## 3) Load data and normalize schema

In [ ]:

df = pd.read_csv(CSV_PRIORS, low_memory=False)

# Parse potential date columns
for c in df.columns:
    if "date" in c.lower():
        df[c] = pd.to_datetime(df[c], errors="coerce")

# Normalize column names -> mu_t, kappa_t, count, coverage, date, mutation, site_id, pi
lower = {c.lower(): c for c in df.columns}
mu_c    = lower.get("mu_t") or lower.get("mu")
kappa_c = lower.get("kappa_t") or lower.get("kappa")
y_c     = lower.get("count") or lower.get("y") or lower.get("success")
n_c     = lower.get("coverage") or lower.get("n") or lower.get("total")
pi_c    = lower.get("pi")  # zero inflation (optional)

date_c  = None
for c in df.columns:
    if "date" in c.lower():
        date_c = c; break

mutation_c = lower.get("mutation")
site_c     = lower.get("site_id")

if mu_c is None or kappa_c is None or y_c is None or n_c is None:
    raise ValueError("priors_full_detail.csv must have mu/mu_t, kappa/kappa_t, count, coverage (and optionally date, mutation, site_id, pi)")

# Ensure expected columns exist with canonical names
df.rename(columns={mu_c:"mu_t", kappa_c:"kappa_t", y_c:"count", n_c:"coverage"}, inplace=True)
if date_c: df.rename(columns={date_c:"date"}, inplace=True)
if mutation_c: df.rename(columns={mutation_c:"mutation"}, inplace=True)
if site_c: df.rename(columns={site_c:"site_id"}, inplace=True)
if pi_c and pi_c != "pi": df.rename(columns={pi_c:"pi"}, inplace=True)

# Fill missing optional cols
if "pi" not in df.columns: df["pi"] = 0.0

# Compute AF if not present
if "af" not in df.columns:
    with np.errstate(divide="ignore", invalid="ignore"):
        df["af"] = np.where(df["coverage"].fillna(0)>0, df["count"]/df["coverage"].clip(lower=1), np.nan)

# Sanitize types and ranges
for c in ["mu_t", "kappa_t", "count", "coverage", "pi"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df = df.dropna(subset=["mu_t","kappa_t","count","coverage","pi"]).copy()

df["count"]    = df["count"].astype(int)
df["coverage"] = df["coverage"].astype(int)
df["count"]    = np.clip(df["count"], 0, df["coverage"])

df["mu_t"]     = np.clip(df["mu_t"].to_numpy(float), 1e-9, 1-1e-9)
df["kappa_t"]  = np.clip(df["kappa_t"].to_numpy(float), EPS, 1e12)
df["pi"]       = np.clip(df["pi"].to_numpy(float), 0.0, 1.0 - 1e-12)

if ROW_LIMIT_GLOBAL and len(df) > ROW_LIMIT_GLOBAL:
    df = df.sample(ROW_LIMIT_GLOBAL, random_state=42).copy()
    print(f"[warn] ROW_LIMIT_GLOBAL applied: using {len(df):,} rows")

print(df[["mu_t","kappa_t","count","coverage","pi"]].describe().T)

## 4) Core math (ZIBB predictive cdf/ppf, PIT, mean/var)

In [ ]:

def _clip01(x, eps=EPS):
    return np.clip(np.asarray(x, float), eps, 1.0 - eps)

def _bb_mean_var(n, mu, kappa):
    # Return mean/var of Beta-Binomial counts.
    mu_c = _clip01(mu); k_c = np.clip(kappa, EPS, np.inf)
    a = mu_c*k_c; b = (1.0 - mu_c)*k_c
    mean = n * mu_c
    var  = n * mu_c * (1.0 - mu_c) * ((a + b + n) / (a + b + 1.0))
    return mean, np.maximum(var, EPS)

def _zibb_mean_var(n, mu, kappa, pi):
    # Mean/var of Zero-Inflated Beta-Binomial counts (mixture with point mass at 0).
    m_bb, v_bb = _bb_mean_var(n, mu, kappa)
    m = (1.0 - pi) * m_bb
    v = (1.0 - pi) * v_bb + (pi * (1.0 - pi)) * (m_bb ** 2)
    return m, np.maximum(v, EPS)

def _cdf_zibb(y, n, mu, kappa, pi):
    # CDF for ZIBB counts at integer y (supports vector inputs).
    mu_c = _clip01(mu); k_c = np.clip(kappa, EPS, 1e12)
    a = mu_c * k_c; b = (1.0 - mu_c) * k_c
    Fy = betabinom.cdf(np.clip(y, -1, None), n, a, b)
    return np.clip(pi + (1.0 - pi) * Fy, 0.0, 1.0)

def _ppf_zibb(q, n, mu, kappa, pi):
    # Exact ZIBB quantile: if q <= pi -> 0, else transform q'=(q-pi)/(1-pi) then Beta-Binomial ppf at q'.
    q = np.asarray(q, float)
    mu_c = _clip01(mu); k_c = np.clip(kappa, EPS, 1e12)
    a = mu_c * k_c; b = (1.0 - mu_c) * k_c
    qprime = (q - pi) / np.maximum(1.0 - pi, EPS)
    qprime = np.clip(qprime, 0.0, 1.0 - EPS)
    y = betabinom.ppf(qprime, n, a, b).astype(np.int64)
    y = np.where(q <= pi, 0, y)
    return np.clip(y, 0, n.astype(np.int64))

def _ppf_chunks(qs, n, mu, kappa, pi, chunk=CHUNK):
    # Chunked multi-quantile ppf for ZIBB.
    qs = np.atleast_1d(np.asarray(qs, float))
    N  = len(n)
    out = np.empty((qs.size, N), dtype=np.int64)
    use_ap = np.zeros(N, dtype=bool)
    if isinstance(USE_APPROX_THRESHOLD_N, (int, float)):
        use_ap = (np.asarray(n) >= int(USE_APPROX_THRESHOLD_N))
    # hybrid: decide once for all qs
    nk = n * kappa
    central = (qs.min() >= EASY_CENTRAL_Q[0]) and (qs.max() <= EASY_CENTRAL_Q[1])
    easy_mask = (nk >= EASY_MIN_NK) & central if HYBRID_INTERVAL else np.zeros(N, dtype=bool)

    for s in range(0, N, chunk):
        e = min(s + chunk, N)
        n_s = n[s:e]; mu_s = mu[s:e]; k_s = kappa[s:e]; pi_s = pi[s:e]
        ap_s = use_ap[s:e] | easy_mask[s:e]

        for i, q in enumerate(qs):
            if ap_s.any():
                # Normal approximation for "easy" rows; exact for the rest
                y = np.empty(e - s, dtype=np.int64)
                if (~ap_s).any():
                    y[~ap_s] = _ppf_zibb(q, n_s[~ap_s], mu_s[~ap_s], k_s[~ap_s], pi_s[~ap_s])
                # normal approx on zibb via inverse CDF of Normal(mean,var)
                m, v = _zibb_mean_var(n_s[ap_s], mu_s[ap_s], k_s[ap_s], pi_s[ap_s])
                sd = np.sqrt(v)
                z = np.sqrt(2.0) * _erfinv(np.clip(2.0*q - 1.0, -1.0 + EPS, 1.0 - EPS))
                y_ap = np.floor(m + z * sd + 0.5).astype(np.int64)
                y[ap_s] = np.clip(y_ap, 0, n_s[ap_s].astype(np.int64))
            else:
                y = _ppf_zibb(q, n_s, mu_s, k_s, pi_s)
            out[i, s:e] = y
    return out if qs.size > 1 else out[0]

def randomized_pit(y, n, mu, kappa, pi):
    # Randomized PIT for discrete predictive: U = F(y-1) + U(0,1)*(F(y)-F(y-1)).
    Fy  = _cdf_zibb(y,   n, mu, kappa, pi)
    Fym = _cdf_zibb(y-1, n, mu, kappa, pi)
    U = np.clip(Fym + np.random.default_rng(SEED).random(y.size) * np.maximum(Fy - Fym, 0.0), 0.0, 1.0)
    return U

## 5) Predictive bands + PIT + Outliers

In [ ]:

def add_predictive_metrics(df, q_lo=PRED_Q_LO, q_hi=PRED_Q_HI):
    n  = df["coverage"].to_numpy(np.int64, copy=False)
    y  = df["count"].to_numpy(np.int64, copy=False)
    mu = df["mu_t"].to_numpy(np.float64, copy=False)
    k  = df["kappa_t"].to_numpy(np.float64, copy=False)
    pi = df["pi"].to_numpy(np.float64, copy=False)
    mask = n > 0

    # Exact counts intervals (hybrid/approx allowed via _ppf_chunks)
    lo = _ppf_chunks(q_lo, n, mu, k, pi, chunk=CHUNK).astype(float)
    hi = _ppf_chunks(q_hi, n, mu, k, pi, chunk=CHUNK).astype(float)
    np.clip(lo, 0, n, out=lo); np.clip(hi, 0, n, out=hi)

    # AF bands: convert from counts
    n_f = n.astype(np.float32 if FAST_FLOAT32_AF else float, copy=False)
    pred_lo_af = np.full_like(n_f, np.nan)
    pred_hi_af = np.full_like(n_f, np.nan)
    np.divide(lo, n_f, out=pred_lo_af, where=mask)
    np.divide(hi, n_f, out=pred_hi_af, where=mask)

    df["pred_lo_ct"] = lo
    df["pred_hi_ct"] = hi
    df["pred_lo_af"] = pred_lo_af
    df["pred_hi_af"] = pred_hi_af

    # PIT
    U = randomized_pit(y, n, mu, k, pi) if RAND_PIT else 0.5 * ( _cdf_zibb(y, n, mu, k, pi) + _cdf_zibb(y-1, n, mu, k, pi) )
    df["pit_u"] = U.astype(np.float32 if FAST_FLOAT32_AF else float, copy=False)

    # Outlier flag
    af = df["af"].to_numpy(np.float32 if FAST_FLOAT32_AF else float, copy=False)
    outlier = (af < pred_lo_af) | (af > pred_hi_af)
    df["outlier"] = np.where(np.isfinite(af), outlier, False)

    print(f"[info] Predictive bands + PIT computed for {len(df):,} rows (RAND_PIT={RAND_PIT}, chunk={CHUNK})")
    return df

df = add_predictive_metrics(df, q_lo=PRED_Q_LO, q_hi=PRED_Q_HI)

## 6) Selection of sites/mutations for per-pair outputs

In [ ]:

sites_all = df["site_id"].dropna().astype(str).unique().tolist() if "site_id" in df.columns else []
muts_all  = df["mutation"].dropna().astype(str).unique().tolist() if "mutation" in df.columns else []
sites_sel = sites_all[:MAX_SITES] if (MAX_SITES is not None) else sites_all
muts_sel  = muts_all[:MAX_MUTS]   if (MAX_MUTS is not None)   else muts_all

if ("site_id" in df.columns) and ("mutation" in df.columns):
    df_sel = df[df["site_id"].isin(sites_sel) & df["mutation"].isin(muts_sel)].copy()
else:
    df_sel = df.copy()

print(f"[info] Selected: {len(sites_sel)}/{len(sites_all)} sites; {len(muts_sel)}/{len(muts_all)} mutations; rows={len(df_sel):,}")

## 7) Plot helpers

In [ ]:

from matplotlib.ticker import MaxNLocator

def _ax_format_time(ax, title=None):
    ax.xaxis.set_major_locator(MaxNLocator(nbins=6))
    ax.set_xlabel("Date"); ax.set_ylabel("Allele frequency")
    if title: ax.set_title(title, fontsize=11)

def _plot_series_panel(ax, d, title=None):
    d_ = d.iloc[::max(1,int(PLOT_EVERY_NTH))].copy()
    ax.fill_between(d_["date"], d_["pred_lo_af"], d_["pred_hi_af"], alpha=0.25, label=f"Pred {int((PRED_Q_HI-PRED_Q_LO)*100)}% AF")
    ax.plot(d_["date"], d_["mu_t"], linewidth=2, label="mu(t)")
    cov = d_["coverage"].fillna(0)
    sz = np.clip(np.sqrt(cov)/4.0, 3, 14)
    ax.scatter(d_["date"], d_["af"], s=sz, alpha=0.7, label="Observed AF")
    out = d_.loc[d_["outlier"]]
    if not out.empty:
        ax.scatter(out["date"], out["af"], s=np.clip(np.sqrt(out["coverage"])/4.0, 3, 14),
                   facecolors="none", edgecolors="r", linewidths=1.0, label="Outlier")
    _ax_format_time(ax, title=title)
    ax.set_ylim(-0.03, 1.03)
    ax.legend(loc="best", fontsize=8, frameon=False)

def save_pair_detail_png(d, site_id, mutation, out_dir: Path):
    fig = plt.figure(figsize=PAGESIZE)
    gs = fig.add_gridspec(2, 2)
    ax1 = fig.add_subplot(gs[0, :])
    ax2 = fig.add_subplot(gs[1, 0])
    ax3 = fig.add_subplot(gs[1, 1])

    _plot_series_panel(ax1, d, title=f"{site_id} - {mutation} (series)")

    d_ = d.iloc[::max(1,int(PLOT_EVERY_NTH))].copy()
    ax2.fill_between(d_["date"], d_["pred_lo_ct"], d_["pred_hi_ct"], alpha=0.25)
    ax2.plot(d_["date"], d_["count"], linewidth=1.5)
    ax2.set_title("Counts vs Predictive band"); ax2.set_ylabel("Count")
    ax2.xaxis.set_major_locator(MaxNLocator(nbins=6))

    u = d["pit_u"].dropna().to_numpy()
    ax3.hist(u, bins=20, range=(0,1), density=True)
    ax3.set_title("PIT (randomized)"); ax3.set_xlabel("u"); ax3.set_ylabel("density")

    fig.tight_layout()
    out = out_dir / f"pair_{safe_fname(site_id)}__{safe_fname(mutation)}.png"
    fig.savefig(out, dpi=DPI)
    plt.close(fig)
    return out

## 8) Generate per-pair PNGs

In [ ]:

pairs_dir = FIG_DIR / "pairs"; pairs_dir.mkdir(parents=True, exist_ok=True)
glob_dir  = FIG_DIR / "global"; glob_dir.mkdir(parents=True, exist_ok=True)
gmt_dir   = FIG_DIR / "global_ts"; gmt_dir.mkdir(parents=True, exist_ok=True)

pairs = df_sel[["site_id","mutation"]].dropna().drop_duplicates().sort_values(["site_id","mutation"]).to_records(index=False) \
         if (("site_id" in df_sel.columns) and ("mutation" in df_sel.columns)) else []
if MAX_PAIRS is not None:
    pairs = pairs[:MAX_PAIRS]

count_pairs = 0
for s, m in pairs:
    d = df_sel[(df_sel["site_id"]==s) & (df_sel["mutation"]==m)].sort_values("date" if "date" in df_sel.columns else df_sel.index.name or df_sel.index)
    if d.empty: 
        continue
    _ = save_pair_detail_png(d, s, m, pairs_dir)
    count_pairs += 1

print(f"[info] Wrote {count_pairs} per-pair PNGs -> {pairs_dir}")

## 9) Global plots

In [ ]:

def _save_hist(data, bins, title, xlabel, out_path):
    fig, ax = plt.subplots(figsize=(10,4))
    ax.hist(data, bins=bins)
    ax.set_title(title); ax.set_xlabel(xlabel); ax.set_ylabel("count")
    fig.savefig(out_path, dpi=DPI); plt.close(fig)

_save_hist(df["mu_t"].dropna().to_numpy(float), bins=50,
           title="Distribution of mu(t)", xlabel="mu", out_path=glob_dir/"mu_distribution.png")
_save_hist(df["kappa_t"].dropna().to_numpy(float), bins=50,
           title="Distribution of kappa", xlabel="kappa", out_path=glob_dir/"kappa_distribution.png")

# PIT global
u = df["pit_u"].dropna().to_numpy()
ks_p = np.nan
if u.size>0:
    ks = kstest(u, "uniform")
    ks_p = float(ks.pvalue)
    fig, ax = plt.subplots(figsize=(8,4))
    ax.hist(u, bins=30, range=(0,1), density=True)
    ax.set_title(f"Global PIT (randomized){'' if np.isnan(ks_p) else f' - KS p={ks_p:.3g}'}")
    ax.set_xlabel("u"); ax.set_ylabel("density")
    fig.savefig(glob_dir/"pit_hist_global.png", dpi=DPI); plt.close(fig)

# Rows per day
if "date" in df.columns:
    rows_per_date = df.groupby("date").size().reset_index(name="n")
    fig, ax = plt.subplots(figsize=(12,4))
    ax.bar(rows_per_date["date"], rows_per_date["n"], width=1.0)
    ax.set_title("Number of rows per day"); ax.set_ylabel("rows")
    ax.xaxis.set_major_locator(MaxNLocator(nbins=8))
    fig.savefig(glob_dir/"rows_per_day.png", dpi=DPI); plt.close(fig)

# kappa vs mu scatter
samp = df[["mu_t","kappa_t"]].dropna()
if len(samp) > 200_000:
    samp = samp.sample(200_000, random_state=123)
fig, ax = plt.subplots(figsize=(6,5))
ax.scatter(samp["mu_t"], samp["kappa_t"], s=6, alpha=0.3)
ax.set_xlabel("mu"); ax.set_ylabel("kappa"); ax.set_title("Scatter of kappa vs mu")
fig.savefig(FIG_DIR/"scatter_kappa_vs_mu.png", dpi=DPI); plt.close(fig)

# Optional: process noise if available
proc = _read_csv_opt(PRIORS_DIR / "process_noise_by_mutation.csv")
if proc is not None and {"q_LL_hat","q_b_hat"}.issubset(proc.columns):
    x = np.clip(proc["q_LL_hat"].to_numpy(float), 1e-12, None)
    y = np.clip(proc["q_b_hat"].to_numpy(float),  1e-12, None)
    fig, ax = plt.subplots(figsize=(5,5))
    ax.scatter(x, y, s=12, alpha=0.7)
    if np.all(x>0) and np.all(y>0):
        ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("q_LL_hat"); ax.set_ylabel("q_b_hat"); ax.set_title("Per-mutation process noise")
    fig.savefig(FIG_DIR/"process_noise_scatter.png", dpi=DPI); plt.close(fig)

    _save_hist(np.log10(x), bins=40, title="Distribution of log10 q_LL_hat", xlabel="log10 q_LL_hat", out_path=FIG_DIR/"q_LL_hist_log10.png")
    _save_hist(np.log10(y), bins=40, title="Distribution of log10 q_b_hat",   xlabel="log10 q_b_hat",   out_path=FIG_DIR/"q_b_hist_log10.png")

# Global mu(t) lines per mutation
gts = _read_csv_opt(PRIORS_DIR / "detail_global_timeseries.csv", parse_dates=["date"])
if gts is not None and {"date","mutation","mu_t"}.issubset(gts.columns):
    gsrc = gts.copy()
elif ("date" in df.columns) and ("mutation" in df.columns):
    gsrc = (df.groupby(["date","mutation"], as_index=False).agg(mu_t=("mu_t","mean")))
else:
    gsrc = pd.DataFrame()

if not gsrc.empty:
    muts = gsrc["mutation"].dropna().astype(str).unique().tolist()
    muts = muts[:MAX_MUTS] if (MAX_MUTS is not None) else muts
    for m in muts:
        d = gsrc[gsrc["mutation"]==m].sort_values("date")
        if d.empty: continue
        fig, ax = plt.subplots(figsize=(12,4))
        ax.plot(d["date"], d["mu_t"], linewidth=2)
        ax.set_ylim(-0.03, 1.03)
        ax.set_title(f"Global mu(t) - {m}")
        ax.set_xlabel("Date"); ax.set_ylabel("mu"); ax.xaxis.set_major_locator(MaxNLocator(nbins=8))
        fig.savefig(gmt_dir / f"global_mu_{safe_fname(m)}.png", dpi=DPI); plt.close(fig)

# Coverage diagnostics by coverage decile (PNG)
cov = df[["coverage","count","pred_lo_ct","pred_hi_ct"]].dropna()
cov = cov[cov["coverage"]>0].copy()
if not cov.empty:
    cov["tile"] = pd.qcut(cov["coverage"], q=10, duplicates="drop")
    cov["covered"] = (cov["count"] >= cov["pred_lo_ct"]) & (cov["count"] <= cov["pred_hi_ct"])
    cov_rate = cov.groupby("tile", observed=True)["covered"].mean().reset_index(name="coverage_rate")
    fig, ax = plt.subplots(figsize=(10,4))
    ax.bar(range(len(cov_rate)), cov_rate["coverage_rate"])
    ax.set_xticks(range(len(cov_rate)))
    ax.set_xticklabels(cov_rate["tile"].astype(str), rotation=45, ha="right")
    ax.set_title(f"Predictive coverage (counts) by coverage decile - target {(PRED_Q_HI-PRED_Q_LO):.0%}")
    ax.set_ylim(0, 1)
    fig.savefig(FIG_DIR / "coverage_by_coverage_decile.png", dpi=DPI); plt.close(fig)

## 10) Metrics utilities

In [ ]:

def _save(df_out: pd.DataFrame, name: str, float_fmt="%.6g"):
    p = METRIC_DIR / name
    df_out.to_csv(p, index=False, float_format=float_fmt)
    print("[csv]", p)

def _wilson(p_hat, n, z=1.96):
    if n <= 0: return (np.nan, np.nan)
    denom = 1 + z**2/n
    center = (p_hat + (z**2)/(2*n)) / denom
    half = z * math.sqrt((p_hat*(1-p_hat)/n) + (z**2)/(4*n**2)) / denom
    return (center - half, center + half)

def _hist_df(x, bins=60, rng=None, variable="value"):
    counts, edges = np.histogram(x, bins=bins, range=rng)
    widths = edges[1:] - edges[:-1]
    total = counts.sum()
    dens = np.zeros_like(counts, float)
    if total > 0:
        p = counts / total
        with np.errstate(divide="ignore", invalid="ignore"):
            dens = np.where(widths > 0, p / widths, 0.0)
    return pd.DataFrame({"variable": variable, "bin_left": edges[:-1], "bin_right": edges[1:],
                         "count": counts, "density": dens})

def _summarize(name, arr):
    s = pd.Series(arr).describe(percentiles=[0.01,0.05,0.10,0.25,0.5,0.75,0.90,0.95,0.99])
    s = s.rename(index={"count":"count","mean":"mean","std":"std","min":"min","1%":"q01","5%":"q05",
                        "10%":"q10","25%":"q25","50%":"q50","75%":"q75","90%":"q90","95%":"q95","99%":"q99","max":"max"})
    out = s.to_frame("value").reset_index().rename(columns={"index":"stat"})
    out.insert(0,"variable",name); return out

def _pearson(x, y):
    r = pearsonr(x, y); return float(getattr(r,"statistic",r[0])), float(getattr(r,"pvalue",r[1]))

def _spearman(x, y):
    r = spearmanr(x, y); return float(getattr(r,"correlation",r[0])), float(getattr(r,"pvalue",r[1]))

def _kendall(x, y):
    r = kendalltau(x, y); return float(getattr(r,"correlation",r[0])), float(getattr(r,"pvalue",r[1]))

def _roc_pr_from_scores(y_true, scores):
    order = np.argsort(scores)
    s = scores[order]; t = y_true[order]
    uniq = np.unique(s)
    if uniq.size > 400:
        qs = np.quantile(s, np.linspace(0, 1, 401))
        thr = np.unique(qs)
    else:
        thr = uniq
    TPR=[]; FPR=[]; PREC=[]; REC=[]
    P = float(np.sum(t==1)); N0 = float(np.sum(t==0))
    if P==0 or N0==0:
        return pd.DataFrame(columns=["threshold","TPR","FPR","precision","recall"]), np.nan, np.nan
    for th in thr[::-1]:
        pred = (scores >= th)
        TP = float(np.sum(pred & (y_true==1)))
        FP = float(np.sum(pred & (y_true==0)))
        FN = P - TP; TN = N0 - FP
        tpr = TP / P; fpr = FP / N0
        prec = TP / max(TP+FP, 1.0); rec = tpr
        TPR.append(tpr); FPR.append(fpr); PREC.append(prec); REC.append(rec)
    auc = 0.0
    if len(FPR) > 1:
        x = np.array(FPR); y = np.array(TPR)
        order = np.argsort(x); auc = float(np.trapz(y[order], x[order]))
    ap = 0.0
    if len(REC) > 1:
        x = np.array(REC); y = np.array(PREC)
        order = np.argsort(x); ap = float(np.trapz(y[order]))
    curve = pd.DataFrame({"threshold":thr[::-1],"TPR":TPR,"FPR":FPR,"precision":PREC,"recall":REC})
    return curve, auc, ap

## 11) Compute metrics (coverage, PIT tests, fit, calibration, detection, LOD, QA)

In [ ]:

t0 = time.time()
y  = df["count"].to_numpy(int)
n  = df["coverage"].to_numpy(int)
mu = df["mu_t"].to_numpy(float)
k  = df["kappa_t"].to_numpy(float)
pi = df["pi"].to_numpy(float)
N  = len(df)
print(f"[metrics] N={N:,}")

# Mean/var and residuals
mean,var = _zibb_mean_var(n, mu, k, pi)
sd = np.sqrt(var)
resid = y - mean
with np.errstate(divide="ignore", invalid="ignore"):
    z = resid / sd

# ---- Coverage at canonical levels ----
LEVELS = [0.50, 0.90, 0.95]
qs = np.asarray(LEVELS, float)
qlo = (1.0 - qs)/2.0; qhi = (1.0 + qs)/2.0
los = _ppf_chunks(qlo, n, mu, k, pi, chunk=CHUNK); his = _ppf_chunks(qhi, n, mu, k, pi, chunk=CHUNK)

cov_rows=[]; width_rows=[]
for i, lvl in enumerate(qs):
    lo, hi = los[i], his[i]
    inside = (y >= lo) & (y <= hi); lower=(y<lo); upper=(y>hi)
    emp=float(inside.mean()); wil_lo,wil_hi=_wilson(emp,N)
    cov_rows.append({"nominal":float(lvl),"empirical":emp,"bias":emp-float(lvl),
                     "lower_miss_rate":float(lower.mean()),"upper_miss_rate":float(upper.mean()),
                     "asymmetry_upper_minus_lower":float(upper.mean()-lower.mean()),
                     "mean_width":float((hi-lo).mean()),"median_width":float(np.median(hi-lo)),
                     "wilson_lo":wil_lo,"wilson_hi":wil_hi,"sample_n":int(N)})
    width_rows.append({"level":float(lvl),"mean_width":float((hi-lo).mean()),"median_width":float(np.median(hi-lo))})

_save(pd.DataFrame(cov_rows), "priors_predictive_coverage.csv")
_save(pd.DataFrame(width_rows), "priors_width_by_level.csv")

# ---- Coverage grid ----
GRID   = [0.50, 0.60, 0.70, 0.80, 0.90, 0.95]
grid = np.asarray(GRID, float)
glo = (1.0 - grid)/2.0; ghi = (1.0 + grid)/2.0
g_los = _ppf_chunks(glo, n, mu, k, pi, chunk=CHUNK); g_his = _ppf_chunks(ghi, n, mu, k, pi, chunk=CHUNK)
grid_rows=[]
for i, lvl in enumerate(grid):
    inside = (y >= g_los[i]) & (y <= g_his[i])
    grid_rows.append({"nominal": float(lvl), "empirical": float(inside.mean()),
                      "bias": float(inside.mean() - lvl), "sample_n": int(N)})
_save(pd.DataFrame(grid_rows), "priors_predictive_coverage_grid.csv")

# ---- Calibration by deciles (mu, log10 kappa, lambda=n*mu) ----
def coverage_by_bins(values, name, levels):
    labels = pd.qcut(values, q=10, duplicates="drop")
    rows=[]
    for i, lvl in enumerate(levels):
        lo = g_los[i]; hi = g_his[i]
        inside = (y >= lo) & (y <= hi)
        g = pd.DataFrame({name: labels, "inside": inside})
        gg = g.groupby(name, dropna=False)["inside"].mean().reset_index()
        for _, r in gg.iterrows():
            rows.append({"bin": str(r[name]), "nominal": float(lvl), "empirical": float(r["inside"]), "variable": name})
    return pd.DataFrame(rows)

lam = n * mu; l10k = np.log10(np.clip(k, EPS, None))
cov_bins = pd.concat([
    coverage_by_bins(mu, "mu_decile", grid),
    coverage_by_bins(l10k, "l10k_decile", grid),
    coverage_by_bins(lam, "lambda_decile", grid),
], ignore_index=True)
_save(cov_bins, "priors_coverage_by_bins.csv")

# ---- PIT (randomized), tests & histogram ----
PIT_BINS = 20
take = np.arange(N) if N <= ROW_SAMPLE_PIT else np.random.default_rng(SEED).choice(N, ROW_SAMPLE_PIT, replace=False)
U = df["pit_u"].to_numpy(float)[take]

hist,edges=np.histogram(U,bins=PIT_BINS,range=(0,1))
expected=U.size/PIT_BINS
ks=kstest(U,"uniform")
chi=chisquare(hist,f_exp=np.full_like(hist,expected,dtype=float))
if cramervonmises is not None:
    cvm=cramervonmises(U,"uniform"); cvm_stat=float(cvm.statistic); cvm_p=float(cvm.pvalue)
else:
    cvm_stat=cvm_p=np.nan

pit_bins = pd.DataFrame({"bin_left":edges[:-1],"bin_right":edges[1:],
                         "count":hist,"density":hist/max(hist.sum(),1),"expected_count_uniform":expected,
                         "sample_n":U.size})
_save(pit_bins, "priors_pit_uniformity_bins.csv")

pit_tests = pd.DataFrame([{
    "test":"KS (Uniform)","statistic":float(getattr(ks,"statistic",ks[0])),"pvalue":float(getattr(ks,"pvalue",ks[1])),
    "N":int(U.size),"bins":PIT_BINS
},{
    "test":"Chi-square (Uniform bins)","statistic":float(getattr(chi,"statistic",chi[0])),"pvalue":float(getattr(chi,"pvalue",chi[1])),
    "N":int(U.size),"bins":PIT_BINS
},{
    "test":"Cramer-von Mises (Uniform)","statistic":cvm_stat,"pvalue":cvm_p,"N":int(U.size),"bins":PIT_BINS
}])
_save(pit_tests, "priors_pit_tests.csv")

# ---- Fit diagnostics ----
with np.errstate(divide="ignore",invalid="ignore"):
    vr=(resid**2)/var
vr_fin=vr[np.isfinite(vr)]
rmsz=float(np.sqrt(np.mean(z**2)))
z_mean=float(np.mean(z)); z_std=float(np.std(z))
z_sk=float(skew(z,nan_policy="omit")); z_ku=float(kurtosis(z,fisher=True,nan_policy="omit"))
try:
    k2=normaltest(z,nan_policy="omit"); k2_stat=float(getattr(k2,"statistic",k2[0])); k2_p=float(getattr(k2,"pvalue",k2[1]))
except Exception:
    k2_stat=k2_p=np.nan

# Average log likelihood under ZIBB
def _mix_logpmf(y, n, mu, kappa, pi):
    mu_c = _clip01(mu); k_c = np.clip(kappa, EPS, np.inf)
    a = mu_c * k_c; b = (1.0 - mu_c) * k_c
    log_bb = betabinom.logpmf(y, n, a, b)
    log1m  = np.log1p(-np.clip(pi, 0.0, 1.0 - EPS))
    out = log1m + log_bb
    z0 = (y == 0)
    if np.any(z0):
        log_pi  = np.log(np.clip(pi[z0], EPS, 1.0 - EPS))
        log_bb0 = betabinom.logpmf(0, n[z0], a[z0], b[z0])
        M = np.vstack([log_pi, log1m[z0] + log_bb0])
        amax = np.max(M, axis=0)
        out[z0] = amax + np.log(np.exp(M[0] - amax) + np.exp(M[1] - amax))
    return out

logpmf = _mix_logpmf(y,n,mu,k,pi)
r_p,p_p=_pearson(y,mean); r_s,p_s=_spearman(y,mean)
cov_ym=float(np.cov(mean,y,ddof=0)[0,1]) if N>1 else np.nan
var_m=float(np.var(mean)) if N>0 else np.nan
slope=float(cov_ym/var_m) if var_m and np.isfinite(var_m) and var_m>0 else np.nan
intercept=float(np.mean(y)-slope*np.mean(mean)) if np.isfinite(slope) else np.nan
r2=float(r_p**2) if np.isfinite(r_p) else np.nan

fit_df = pd.DataFrame([{
    "N":int(N),
    "rmse":float(np.sqrt(np.mean(resid**2))), "mae":float(np.mean(np.abs(resid))),
    "rmsz":rmsz, "avg_logpmf":float(np.mean(logpmf[np.isfinite(logpmf)])),
    "variance_ratio_mean":float(np.mean(vr_fin)),
    "variance_ratio_median":float(np.median(vr_fin)),
    "variance_ratio_q05":float(np.quantile(vr_fin,0.05)),
    "variance_ratio_q95":float(np.quantile(vr_fin,0.95)),
    "z_mean":z_mean, "z_std":z_std, "z_skew":z_sk, "z_kurtosis_fisher":z_ku,
    "normaltest_k2_stat":k2_stat, "normaltest_k2_pvalue":k2_p,
    "pearson_y_mean_r":r_p, "pearson_y_mean_p":p_p,
    "spearman_y_mean_rho":r_s, "spearman_p":p_s,
    "regress_y_on_mean_slope":slope, "regress_y_on_mean_intercept":intercept, "regress_y_on_mean_r2":r2
}])
_save(fit_df, "priors_fit_diagnostics.csv")

# ---- Variable summaries & histograms ----
var_summary = pd.concat([
    _summarize("y",y), _summarize("n",n), _summarize("mu",mu), _summarize("kappa",k), _summarize("pi",pi),
    _summarize("mean",mean), _summarize("var",var), _summarize("sd",np.sqrt(var)),
    _summarize("z",z), _summarize("resid",resid), _summarize("u_pit",df["pit_u"].to_numpy(float)),
    _summarize("variance_ratio",vr), _summarize("logpmf",logpmf[np.isfinite(logpmf)]),
], ignore_index=True)
_save(var_summary, "priors_variable_summaries.csv")

z_lo,z_hi = float(np.nanquantile(z,0.001)), float(np.nanquantile(z,0.999))
vr_lo,vr_hi = float(np.nanquantile(vr_fin,0.0)), float(np.nanquantile(vr_fin,0.995))
h = pd.concat([
    _hist_df(mu,bins=60,rng=(0,1),variable="mu"),
    _hist_df(np.log10(np.clip(k, EPS, None)),bins=60,
             rng=(float(np.nanmin(np.log10(np.clip(k, EPS, None)))), float(np.nanmax(np.log10(np.clip(k, EPS, None))))),
             variable="log10_kappa"),
    _hist_df(np.clip(z,z_lo,z_hi),bins=60,rng=(z_lo,z_hi),variable="z"),
    _hist_df(df["pit_u"].to_numpy(float),bins=60,rng=(0,1),variable="u_pit"),
    _hist_df(np.clip(vr,vr_lo,vr_hi),bins=60,rng=(vr_lo,vr_hi),variable="variance_ratio"),
], ignore_index=True)
_save(h, "priors_histograms.csv")

# ---- Correlations ----
r_pk,p_pk=_pearson(mu,k); r_sk,p_sk=_spearman(mu,k); r_kk,p_kk=_kendall(mu,k)
_save(pd.DataFrame([{
    "pearson_mu_kappa":r_pk,"pearson_p":p_pk,
    "spearman_mu_kappa":r_sk,"spearman_p":p_sk,
    "kendall_mu_kappa":r_kk,"kendall_p":p_kk
}]), "priors_mu_kappa_correlation.csv")
_save(pd.DataFrame([{
    "pearson_y_mean":r_p,"pearson_p":p_p,
    "spearman_y_mean":r_s,"spearman_p":p_s,
    "slope_y_on_mean":slope,"intercept_y_on_mean":intercept,"r2":r2
}]), "priors_y_mean_correlation.csv")

# ---- PIT by site/mutation (KS) ----
if "site_id" in df.columns:
    rows=[]
    tmp=pd.DataFrame({"site_id":df["site_id"].astype(str),"U":df["pit_u"].to_numpy(float)})
    for sid,grp in tmp.groupby("site_id"):
        if len(grp)<10: continue
        r=kstest(grp["U"].to_numpy(float),"uniform")
        rows.append({"site_id":sid,"n":int(len(grp)),"ks_stat":float(getattr(r,"statistic",r[0])),"ks_pvalue":float(getattr(r,"pvalue",r[1]))})
    if rows: _save(pd.DataFrame(rows).sort_values("ks_stat",ascending=False), "priors_pit_ks_by_site.csv")

if "mutation" in df.columns and df["mutation"].nunique() <= 1000:
    rows=[]
    tmpm=pd.DataFrame({"mutation":df["mutation"].astype(str),"U":df["pit_u"].to_numpy(float)})
    for m,grp in tmpm.groupby("mutation"):
        if len(grp)<10: continue
        r=kstest(grp["U"].to_numpy(float),"uniform")
        rows.append({"mutation":m,"n":int(len(grp)),"ks_stat":float(getattr(r,"statistic",r[0])),"ks_pvalue":float(getattr(r,"pvalue",r[1]))})
    if rows: _save(pd.DataFrame(rows).sort_values("ks_stat",ascending=False), "priors_pit_ks_by_mutation.csv")

# ---- Extreme misses (outside 99%) ----
lo99=_ppf_chunks(0.005, n, mu, k, pi, chunk=CHUNK); hi99=_ppf_chunks(0.995, n, mu, k, pi, chunk=CHUNK)
miss=(y<lo99)|(y>hi99); idx=np.where(miss)[0]
if idx.size>0:
    take=idx if idx.size<=50000 else np.random.default_rng(123).choice(idx,50000,replace=False)
    miss_df=pd.DataFrame({"y":y[take],"n":n[take],"mu":mu[take],"kappa":k[take],"pi":pi[take],
                          "mean":mean[take],"sd":sd[take],"z":z[take],
                          "y_lo_99":lo99[take],"y_hi_99":hi99[take]})
    for opt in ["site_id","mutation","date"]:
        if opt in df.columns: miss_df[opt]=df[opt].astype(str).values[take]
    _save(miss_df, "priors_outside_99_sample.csv")

# ---- Detection metrics (C>=C_MIN) ----
present = (y >= C_MIN).astype(int)
p_detect = 1.0 - _cdf_zibb(C_MIN - 1, n, mu, k, pi)
curve, auc, ap = _roc_pr_from_scores(present, p_detect)
if not curve.empty: _save(curve, "priors_detection_curve.csv")
brier = float(np.mean((p_detect - present)**2)) if present.size else np.nan
det_summary = pd.DataFrame([{"C_min":C_MIN, "ROC_AUC":auc, "PR_AUC":ap, "Brier":brier,
                             "positive_rate": float(present.mean()), "N": int(N)}])
_save(det_summary, "priors_detection_summary.csv")

# ---- LOD (binary search on mu given group medians) ----
def lod_mu_for_row(n_i, k_i, pi_i, c_min=C_MIN, power=POWER, tol=1e-4, max_iter=50):
    lo, hi = 1e-6, 0.5
    for _ in range(max_iter):
        mid = 0.5*(lo+hi)
        # compute P(Y>=c_min | mu=mid, k_i, pi_i, n_i)
        p_ge = 1.0 - _cdf_zibb(c_min-1, np.array([n_i]), np.array([mid]), np.array([k_i]), np.array([pi_i]))[0]
        if p_ge >= power: hi = mid
        else: lo = mid
        if (hi - lo) < tol: break
    return hi

rows = []
if "mutation" in df.columns:
    for mut, g in df.groupby("mutation", sort=False):
        n_star = int(np.median(g["coverage"])) if LOD_USE_MEDIAN_COVERAGE else int(np.max(g["coverage"]))
        k_star = float(np.median(g["kappa_t"])); pi_star = float(np.median(g["pi"]))
        lod = lod_mu_for_row(n_star, k_star, pi_star)
        rows.append({"group":"mutation","key":str(mut),"n_star":n_star,"kappa_star":k_star,"pi_star":pi_star,"LOD_mu":lod})
if "site_id" in df.columns:
    for sid, g in df.groupby("site_id", sort=False):
        n_star = int(np.median(g["coverage"])) if LOD_USE_MEDIAN_COVERAGE else int(np.max(g["coverage"]))
        k_star = float(np.median(g["kappa_t"])); pi_star = float(np.median(g["pi"]))
        lod = lod_mu_for_row(n_star, k_star, pi_star)
        rows.append({"group":"site_id","key":str(sid),"n_star":n_star,"kappa_star":k_star,"pi_star":pi_star,"LOD_mu":lod})
if rows: _save(pd.DataFrame(rows), "priors_lod_summaries.csv")

# ---- QA scorecard ----
qa=[]
cov_df=pd.DataFrame(cov_rows)
for _,r in cov_df.iterrows():
    nom=float(r["nominal"]); emp=float(r["empirical"]); bias=float(r["bias"]); asym=float(r["asymmetry_upper_minus_lower"])
    tol=0.02 if nom>=0.9 else 0.03; asym_tol=0.02 if nom>=0.9 else 0.03
    status="PASS" if (abs(bias)<=tol and abs(asym)<=asym_tol) else "MINOR"
    if abs(bias)>tol+0.02 or abs(asym)>asym_tol+0.02: status="FLAG"
    qa.append({"module":"Coverage","metric":f"{nom:.2f}","value":f"emp={emp:.4f}, bias={bias:.4f}, asym={asym:.4f}","status":status})

ks_stat=float(getattr(ks,"statistic",ks[0])); ks_p=float(getattr(ks,"pvalue",ks[1]))
ks_status="PASS" if ks_stat<=0.03 else ("MINOR" if ks_stat<=0.05 else "FLAG")
qa.append({"module":"PIT","metric":"KS","value":f"KS={ks_stat:.4f} (p={ks_p:.2e})","status":ks_status})

vr_mean=float(np.mean(vr_fin)); z_mean_ok=abs(float(np.mean(z)))<=0.05
z_std_ok=0.95<=float(np.std(z))<=1.05; rmsz_ok=0.95<=float(np.sqrt(np.mean(z**2)))<=1.05
vrm_ok=0.9<=vr_mean<=1.1
fit_status="PASS" if all([z_mean_ok,z_std_ok,rmsz_ok,vrm_ok]) else ("MINOR" if sum([z_mean_ok,z_std_ok,rmsz_ok,vrm_ok])>=3 else "FLAG")
qa.append({"module":"Fit","metric":"z/VR","value":f"z_mean={float(np.mean(z)):.3f}, z_std={float(np.std(z)):.3f}, RMSZ={float(np.sqrt(np.mean(z**2))):.3f}, VR_mean={vr_mean:.3f}","status":fit_status})

if not math.isnan(auc):
    det_status="PASS" if auc>=0.85 else ("MINOR" if auc>=0.75 else "FLAG")
    qa.append({"module":"Detection","metric":"ROC-AUC","value":f"auc={auc:.3f}, pr_auc={ap:.3f}, brier={brier:.4f}","status":det_status})

_save(pd.DataFrame(qa), "priors_qa_summary.csv")

print("[done]", json.dumps({"N":int(N), "metric_dir": METRIC_DIR.as_posix(), "sec": round(time.time()-t0,2)}))

## 12) Notes
- Quantiles and CDF are computed exactly for the ZIBB unless `HYBRID_INTERVAL=True` and the row meets the
  easy central-quantile and large-`n*kappa` criteria, in which case a normal approximation is used
  for speed. Toggle these knobs at the top if you want pure exactness everywhere.
- The PIT uses randomized definition (recommended for discrete predictive checks).
- All PNGs and CSVs are produced under `results/priors/figures` and `results/priors/metric`.